# 🏥 HealLens AI — Disease Detection CNN Training
### Training a MobileNetV2 model for 7 diseases across Chest, Bone, and Skin categories

**Diseases Covered:**
- 🫁 **Chest:** Pneumonia, Tuberculosis, COVID-19
- 🦴 **Bone:** Fracture, Arthritis
- 🧴 **Skin:** Skin Infection, Psoriasis/Rash

**Steps:**
1. Setup & install dependencies
2. Upload your Kaggle API key
3. Download datasets (500–1000 images per disease)
4. Preprocess & augment images
5. Train MobileNetV2 (Transfer Learning)
6. Evaluate model
7. Export to TensorFlow.js format
8. Download model files for your HealLens app

> ⚡ **Runtime:** Go to `Runtime > Change runtime type > T4 GPU` before running!

## ✅ Step 1: Install Dependencies

In [ ]:
!pip install -q tensorflowjs kaggle pillow matplotlib scikit-learn
print('✅ All dependencies installed!')

## ✅ Step 2: Upload Kaggle API Key

1. Go to https://www.kaggle.com → Your Profile → Settings → API → **Create New Token**
2. This downloads `kaggle.json` to your computer
3. Run the cell below and upload that file

In [ ]:
from google.colab import files
import os

print('📁 Upload your kaggle.json file...')
uploaded = files.upload()

# Move to correct location
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle API key configured!')

## ✅ Step 3: Download All Disease Datasets from Kaggle
Downloading 500–1000 images per disease class

In [ ]:
import os, shutil, random
from pathlib import Path

BASE_DIR = Path('/content/heallens_dataset')
TARGET_PER_CLASS = 800  # 500–1000 images per disease

# Create class directories
CLASSES = [
    'pneumonia', 'tuberculosis', 'covid19',
    'fracture', 'arthritis',
    'skin_infection', 'psoriasis'
]
for cls in CLASSES:
    (BASE_DIR / cls).mkdir(parents=True, exist_ok=True)

print('✅ Dataset directory structure created!')
print(f'📊 Target: {TARGET_PER_CLASS} images per class × {len(CLASSES)} classes = {TARGET_PER_CLASS * len(CLASSES):,} images total')

In [ ]:
# ─── CHEST: Pneumonia Dataset ─────────────────────────────────────────────────
print('⬇️  Downloading Pneumonia dataset...')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /tmp/pneumonia --unzip -q

# Collect all pneumonia images
pneu_dir = Path('/tmp/pneumonia/chest_xray')
pneu_images = []
for split in ['train', 'val', 'test']:
    pneu_images += list((pneu_dir / split / 'PNEUMONIA').glob('*.jpeg'))
    pneu_images += list((pneu_dir / split / 'PNEUMONIA').glob('*.jpg'))
    pneu_images += list((pneu_dir / split / 'PNEUMONIA').glob('*.png'))

random.shuffle(pneu_images)
for i, img in enumerate(pneu_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'pneumonia' / f'pneumonia_{i:04d}{img.suffix}')

print(f'✅ Pneumonia: {len(list((BASE_DIR / "pneumonia").iterdir()))} images collected')

In [ ]:
# ─── CHEST: Tuberculosis Dataset ──────────────────────────────────────────────
print('⬇️  Downloading Tuberculosis dataset...')
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset -p /tmp/tb --unzip -q

tb_dir = Path('/tmp/tb')
tb_images = list(tb_dir.rglob('*.png')) + list(tb_dir.rglob('*.jpg')) + list(tb_dir.rglob('*.jpeg'))
# Filter to TB positives only
tb_images = [p for p in tb_images if 'Tuberculosis' in str(p) or 'TB' in str(p.parent.name)]
if len(tb_images) < 100:
    tb_images = list(tb_dir.rglob('*.png')) + list(tb_dir.rglob('*.jpg'))  # fallback: all

random.shuffle(tb_images)
for i, img in enumerate(tb_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'tuberculosis' / f'tb_{i:04d}{img.suffix}')

print(f'✅ Tuberculosis: {len(list((BASE_DIR / "tuberculosis").iterdir()))} images collected')

In [ ]:
# ─── CHEST: COVID-19 Dataset ──────────────────────────────────────────────────
print('⬇️  Downloading COVID-19 Chest X-Ray dataset...')
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /tmp/covid --unzip -q

covid_dir = Path('/tmp/covid')
covid_images = []
for subdir in covid_dir.rglob('COVID'):
    if subdir.is_dir():
        covid_images += list(subdir.glob('*.png')) + list(subdir.glob('*.jpg'))
if len(covid_images) < 100:
    covid_images = list(covid_dir.rglob('*.png')) + list(covid_dir.rglob('*.jpg'))

random.shuffle(covid_images)
for i, img in enumerate(covid_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'covid19' / f'covid_{i:04d}{img.suffix}')

print(f'✅ COVID-19: {len(list((BASE_DIR / "covid19").iterdir()))} images collected')

In [ ]:
# ─── BONE: Fracture Dataset ───────────────────────────────────────────────────
print('⬇️  Downloading Bone Fracture dataset...')
!kaggle datasets download -d bmadushanirodrigo/fracture-multi-region-x-ray-data -p /tmp/fracture --unzip -q

frac_dir = Path('/tmp/fracture')
frac_images = []
for subdir in frac_dir.rglob('*'):
    if subdir.is_dir() and ('fractured' in subdir.name.lower() or 'fracture' in subdir.name.lower()):
        frac_images += list(subdir.glob('*.jpg')) + list(subdir.glob('*.png')) + list(subdir.glob('*.jpeg'))
if len(frac_images) < 100:
    frac_images = list(frac_dir.rglob('*.jpg')) + list(frac_dir.rglob('*.png'))

random.shuffle(frac_images)
for i, img in enumerate(frac_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'fracture' / f'fracture_{i:04d}{img.suffix}')

print(f'✅ Fracture: {len(list((BASE_DIR / "fracture").iterdir()))} images collected')

In [ ]:
# ─── BONE: Arthritis Dataset ──────────────────────────────────────────────────
print('⬇️  Downloading Knee Arthritis/Osteoarthritis dataset...')
!kaggle datasets download -d shashwatwork/knee-osteoarthritis-dataset-with-severity -p /tmp/arthritis --unzip -q

arth_dir = Path('/tmp/arthritis')
arth_images = list(arth_dir.rglob('*.png')) + list(arth_dir.rglob('*.jpg')) + list(arth_dir.rglob('*.jpeg'))
# Prefer grades 2-4 (clearer arthritis signs)
arth_filtered = [p for p in arth_images if any(g in str(p) for g in ['grade_2', 'grade_3', 'grade_4', 'Grade_2', 'Grade_3', 'Grade_4', '2', '3', '4'])]
if len(arth_filtered) > TARGET_PER_CLASS:
    arth_images = arth_filtered

random.shuffle(arth_images)
for i, img in enumerate(arth_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'arthritis' / f'arthritis_{i:04d}{img.suffix}')

print(f'✅ Arthritis: {len(list((BASE_DIR / "arthritis").iterdir()))} images collected')

In [ ]:
# ─── SKIN: Skin Infection / Fungal Dataset ────────────────────────────────────
print('⬇️  Downloading Skin Disease dataset...')
!kaggle datasets download -d shubhamgoel27/dermnet -p /tmp/skin --unzip -q

skin_dir = Path('/tmp/skin')
skin_images = []
# Target infection categories
infection_keywords = ['bacterial', 'fungal', 'impetigo', 'cellulitis', 'folliculitis', 'tinea', 'infection', 'warts']
for subdir in skin_dir.rglob('*'):
    if subdir.is_dir() and any(kw in subdir.name.lower() for kw in infection_keywords):
        skin_images += list(subdir.glob('*.jpg')) + list(subdir.glob('*.png')) + list(subdir.glob('*.jpeg'))
if len(skin_images) < 200:
    skin_images = list(skin_dir.rglob('*.jpg'))[:TARGET_PER_CLASS * 2]

random.shuffle(skin_images)
for i, img in enumerate(skin_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'skin_infection' / f'skin_inf_{i:04d}{img.suffix}')

print(f'✅ Skin Infection: {len(list((BASE_DIR / "skin_infection").iterdir()))} images collected')

In [ ]:
# ─── SKIN: Psoriasis / Rash Dataset ──────────────────────────────────────────
print('⬇️  Collecting Psoriasis/Rash images from dermnet dataset...')

psoriasis_images = []
psoriasis_keywords = ['psoriasis', 'eczema', 'rash', 'dermatitis', 'urticaria', 'hives', 'erythema']
for subdir in skin_dir.rglob('*'):
    if subdir.is_dir() and any(kw in subdir.name.lower() for kw in psoriasis_keywords):
        psoriasis_images += list(subdir.glob('*.jpg')) + list(subdir.glob('*.png')) + list(subdir.glob('*.jpeg'))

if len(psoriasis_images) < 200:
    # Fallback: ISIC skin lesion dataset
    print('  → Fetching additional psoriasis data...')
    !kaggle datasets download -d riyaelizashaju/skin-disease-classification-image-dataset -p /tmp/psoriasis --unzip -q 2>/dev/null || true
    psoriasis_dir = Path('/tmp/psoriasis')
    for subdir in psoriasis_dir.rglob('*'):
        if subdir.is_dir() and any(kw in subdir.name.lower() for kw in psoriasis_keywords + ['lesion', 'pigmented']):
            psoriasis_images += list(subdir.glob('*.jpg')) + list(subdir.glob('*.png'))
    if len(psoriasis_images) < 100:
        psoriasis_images = list(psoriasis_dir.rglob('*.jpg')) + list(psoriasis_dir.rglob('*.png'))

random.shuffle(psoriasis_images)
for i, img in enumerate(psoriasis_images[:TARGET_PER_CLASS]):
    shutil.copy(img, BASE_DIR / 'psoriasis' / f'psoriasis_{i:04d}{img.suffix}')

print(f'✅ Psoriasis/Rash: {len(list((BASE_DIR / "psoriasis").iterdir()))} images collected')

In [ ]:
# ─── Dataset Summary ──────────────────────────────────────────────────────────
print('\n📊 DATASET SUMMARY')
print('='*45)
total = 0
for cls in CLASSES:
    count = len(list((BASE_DIR / cls).iterdir()))
    total += count
    status = '✅' if count >= 500 else '⚠️ '
    print(f'{status} {cls:<20} {count:>5} images')
print('='*45)
print(f'   TOTAL: {total:,} images')

## ✅ Step 4: Build TF Datasets with Augmentation

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight

print(f'🔧 TensorFlow version: {tf.__version__}')
print(f'🖥️  GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE  = 224   # MobileNetV2 native input size
BATCH     = 32
SEED      = 42

# ──  Load from directory with 80/20 train-val split ──────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.20,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.20,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical'
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f'\n🏷️  Classes ({NUM_CLASSES}): {CLASS_NAMES}')

In [ ]:
# ─── Data Augmentation Pipeline ───────────────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.12),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.10),
], name='data_augmentation')

# MobileNetV2 preprocessing (normalise to [-1, 1])
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def prepare_train(img, lbl):
    img = data_augmentation(img, training=True)
    img = preprocess(img)
    return img, lbl

def prepare_val(img, lbl):
    img = preprocess(img)
    return img, lbl

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(prepare_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(prepare_val,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print('✅ Dataset pipelines ready with augmentation!')

In [ ]:
# ─── Preview Training Samples ─────────────────────────────────────────────────
raw_train = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR, validation_split=0.20, subset='training',
    seed=SEED, image_size=(IMG_SIZE, IMG_SIZE), batch_size=9, label_mode='int'
)
plt.figure(figsize=(12, 8))
for images, labels in raw_train.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(CLASS_NAMES[labels[i]], fontsize=10)
        plt.axis('off')
plt.suptitle('Training Sample Preview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Sample preview done!')

## ✅ Step 5: Build MobileNetV2 Model (Transfer Learning)

In [ ]:
# ─── Load MobileNetV2 base (ImageNet pre-trained, frozen) ────────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze during phase 1

# ─── Build full classification head ───────────────────────────────────────────
inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(256, activation='relu')(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dropout(0.40)(x)
x       = tf.keras.layers.Dense(128, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.20)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='HealLens_MobileNetV2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()
print(f'\n✅ Model built! Trainable params: {model.trainable_variables.__len__():,}')

## ✅ Step 6: Phase 1 Training — Frozen Base (10 Epochs)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint('/content/heallens_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=3, min_lr=1e-6, verbose=1)
]

print('🚀 Phase 1: Training classification head (base frozen)...')
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)
print(f'\n✅ Phase 1 complete! Best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## ✅ Step 7: Phase 2 — Fine-Tuning (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 40 layers of MobileNetV2 for fine-tuning
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),  # much lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print(f'🔓 Fine-tuning layers {fine_tune_at}+ of MobileNetV2')
print(f'   Trainable params: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

print('\n🚀 Phase 2: Fine-tuning...')
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)
print(f'\n✅ Phase 2 complete! Best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## ✅ Step 8: Evaluate Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Reload best weights
model.load_weights('/content/heallens_best.h5')

print('📊 Evaluating on validation set...')
loss, acc, auc = model.evaluate(val_ds, verbose=0)
print(f'\n🎯 FINAL RESULTS:')
print(f'   Accuracy : {acc*100:.2f}%')
print(f'   AUC      : {auc:.4f}')
print(f'   Loss     : {loss:.4f}')

In [ ]:
# Confusion Matrix
y_true, y_pred = [], []
for imgs, lbls in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('HealLens — Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Training History Plot
all_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(all_acc, 'b-o', markersize=4, label='Val Accuracy')
ax1.axvline(x=len(history1.history['val_accuracy']), color='r', linestyle='--', label='Fine-tuning start')
ax1.set_title('Validation Accuracy', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(all_loss, 'r-o', markersize=4, label='Val Loss')
ax2.axvline(x=len(history1.history['val_loss']), color='r', linestyle='--', label='Fine-tuning start')
ax2.set_title('Validation Loss', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('HealLens Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ✅ Step 9: Export to TensorFlow.js Format

In [ ]:
import tensorflowjs as tfjs
import json

TFJS_DIR = '/content/heallens_tfjs_model'

# Save in SavedModel format first, then convert
model.save('/content/heallens_savedmodel')

# Convert to TFJS
tfjs.converters.save_keras_model(model, TFJS_DIR)

print(f'✅ TensorFlow.js model exported to: {TFJS_DIR}')
!ls -lh {TFJS_DIR}

In [ ]:
# Save class names + model metadata for HealLens app
metadata = {
    "classes": CLASS_NAMES,
    "numClasses": NUM_CLASSES,
    "imageSize": IMG_SIZE,
    "modelName": "HealLens_MobileNetV2",
    "accuracy": round(acc * 100, 2),
    "bodyPartMap": {
        "pneumonia":     "chest",
        "tuberculosis":  "chest",
        "covid19":       "chest",
        "fracture":      "bone",
        "arthritis":     "bone",
        "skin_infection":"skin",
        "psoriasis":     "skin"
    },
    "diseaseNameMap": {
        "pneumonia":     "Pneumonia",
        "tuberculosis":  "Tuberculosis",
        "covid19":       "COVID-19",
        "fracture":      "Fracture",
        "arthritis":     "Arthritis",
        "skin_infection":"Skin Infection",
        "psoriasis":     "Psoriasis/Rash"
    }
}

with open(f'{TFJS_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ metadata.json saved!')
print(json.dumps(metadata, indent=2))

## ✅ Step 10: Package & Download Model Files

In [ ]:
import zipfile

zip_path = '/content/heallens_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path(TFJS_DIR).iterdir():
        zf.write(f, f.name)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'✅ Model packaged: heallens_model.zip ({size_mb:.1f} MB)')

print('\n📁 Files in package:')
with zipfile.ZipFile(zip_path, 'r') as zf:
    for info in zf.infolist():
        print(f'  {info.filename:40s} {info.file_size/1024:.1f} KB')

In [ ]:
# Download everything!
from google.colab import files

print('⬇️  Downloading model...')
files.download('/content/heallens_model.zip')

print('\n🎉 SUCCESS! Training complete!')
print('='*55)
print(f'  Model Accuracy : {acc*100:.2f}%')
print(f'  AUC Score      : {auc:.4f}')
print(f'  Diseases       : {NUM_CLASSES} classes')
print(f'  Classes        : {CLASS_NAMES}')
print('='*55)
print('\n📋 NEXT STEPS:')
print('  1. Extract heallens_model.zip')
print('  2. Copy all files to: pdd/model/')
print('  3. The updated ai-engine.js will load them automatically!')